# Cicero Digital – Session 3 (Teil): spaCy + LatinCy

In Session 2 hast du aus den TEI-XML-Dateien Metadaten pro Brief extrahiert und als CSV exportiert.

**In diesem Notebook** laden wir diese CSV wieder ein und führen anschliessend eine einfache sprachliche Vorverarbeitung durch:
- Satzsegmentierung
- Tokenisierung
- Wortartenannotation (POS)
- syntaktisches Parsing (Dependenzen)

Wir halten die Code-Beispiele bewusst einfach. Du sollst die einzelnen Schritte verstehen und anpassen können.

> **Input-Datei:** `cicero_letters.csv`  
> **Voraussetzung:** Die Perseus-TEI-Dateien liegen im Ordner `data/` (wie in Session 2).


## 0. Setup

Wir benötigen `pandas` (Tabellen) und `lxml` (XML). Für die linguistische Analyse nutzen wir `spaCy`.

Zusätzlich **kann** `LatinCy` verwendet werden. Weil Installationen je nach Umgebung variieren, ist der Code unten so geschrieben, dass er **Fallbacks** hat:
1) Wenn `latincy` verfügbar ist, wird es verwendet.
2) Sonst wird ein spaCy-Lateinmodell (`la_core_web_sm`/`la_core_web_lg`) versucht.
3) Sonst wird ein minimales `spacy.blank("la")` genutzt (dann aber ohne POS/Parsing).


In [ ]:
# Basis-Imports
from pathlib import Path
import re
import pandas as pd

from lxml import etree

import spacy
from spacy import displacy


### Optional: Installation (falls nötig)

Wenn du das Notebook lokal (oder in einer Umgebung mit `pip`) ausführst, kannst du folgende Zellen *einmalig* ausführen.

In JupyterLite (Browser) kann `pip` je nach Setup eingeschränkt sein.


In [ ]:
# OPTIONAL (nur falls nötig): spaCy + LatinCy installieren, respl. LatinCy herunterladen
#!pip install https://huggingface.co/latincy/la_core_web_lg/resolve/main/la_core_web_lg-any-py3-none-any.whl

# OPTIONAL (nur falls nötig): spaCy-Lateinmodell installieren
# !python -m spacy download la_core_web_sm

## 1. Brief-CSV einlesen

Wir laden das in Session 2 erzeugte Mini-Dataset. Es enthält pro Brief genau eine Zeile.


In [ ]:
csv_path = Path("outputs") / "cicero_letters.csv"
df = pd.read_csv(csv_path)
df.head()

Ein schneller Check:
- Welche Subkorpora sind enthalten?
- Wie viele Briefe pro Subkorpus?


In [ ]:
df["corpus"].value_counts()


Ein kleiner Qualitätscheck:
- Wie lang sind die Texte (sehr grob, Zeichenlänge)?


In [ ]:
(df["text"].str.len() == 0).mean(), df["text"].str.len().describe()


## 3. LatinCy initialisieren

Wir initialisieren ein NLP-Pipeline-Objekt `nlp`.

- Wenn `latincy` verfügbar ist, nutzen wir es.
- Sonst: wir versuchen ein blanken spaCy.

Wenn du kein Modell installiert hast, bekommst du ein minimales Pipeline-Objekt.
Dann funktionieren **Tokenisierung** und (mit Sentencizer) **Satzsegmentierung**, aber **kein** POS/Parsing.


In [ ]:
def load_nlp():
    try:
        return spacy.load("la_core_web_lg")
    except Exception:
        print("No LatinCy model, initialising blank model")
        nlp = spacy.blank("la")
        if "sentencizer" not in nlp.pipe_names:
            nlp.add_pipe("sentencizer")
        return nlp

nlp = load_nlp()
nlp.pipe_names

## 4. Ein Brief als Beispiel: Token, POS, Dependenzen

Wir wählen einen Brief aus (z.B. erster Brief aus `ad_atticum`) und schauen uns die Analyse an.


In [ ]:
example = df[df["corpus"] == "ad_atticum"].iloc[0]
example[["corpus", "book_n", "letter_n", "date_when"]], example["text"][:300]


In [ ]:
doc = nlp(example["text"])

# Erste Tokens anschauen
[(t.text, t.lemma_, t.pos_, t.dep_, t.head) for t in doc[:25]]


In [ ]:
type(doc)

In [ ]:
list(doc.sents)[0]

In [ ]:
doc[0].__dir__()

### Dependency-Visualisierung (Parsingbaum)

Wenn das Modell Dependenzen liefert, kannst du sie so anzeigen:


In [ ]:
# In Jupyter wird das als SVG gerendert.
displacy.render(list(doc.sents)[0], style="dep", jupyter=True)


In [ ]:
# Anzeigen der von LatinCy markierten Entitäten
displacy.render(list(doc.sents)[:10], style="ent", jupyter=True)

In [ ]:
# Speichern des Parsing-Baumes
with open('parsing-tree.svg', 'w') as outfile:
    outfile.write(displacy.render(list(doc.sents)[0], style="dep", jupyter=False))


## 5. POS-Verteilung: gesamt und nach Subkorpus

Wir zählen POS-Tags über viele Briefe. Für Geschwindigkeit kannst du mit einer Stichprobe arbeiten.


In [ ]:
# Stichprobe (optional): z.B. 200 Briefe für schnelle Runs
sample = df.sample(n=min(200, len(df)), random_state=42)

def pos_counts(texts):
    counts = {}
    for doc in nlp.pipe(texts, batch_size=20):
        for tok in doc:
            if tok.is_space:
                continue
            pos = tok.pos_ if tok.pos_ else "?"
            counts[pos] = counts.get(pos, 0) + 1
    return counts

pos_total = pos_counts(sample["text"].tolist())
pd.Series(pos_total).sort_values(ascending=False).head(15)


### Visualisierung

In [ ]:
import matplotlib.pyplot as plt

pos_series = pd.Series(pos_total).sort_values(ascending=False)
ax = pos_series.head(12).plot(kind="bar")
ax.set_title("POS-Verteilung (Stichprobe)")
ax.set_xlabel("POS")
ax.set_ylabel("Anzahl Tokens")
plt.show()


### POS nach Subkorpus

In [ ]:
rows = []
for corpus, sub in sample.groupby("corpus"):
    counts = pos_counts(sub["text"].tolist())
    s = pd.Series(counts)
    s.name = corpus
    rows.append(s)

pos_by_corpus = pd.DataFrame(rows).fillna(0).astype(int)
pos_by_corpus


In [ ]:
pos_share = pos_by_corpus.div(pos_by_corpus.sum(axis=1), axis=0)
pos_share.head()


In [ ]:
# Beispiel: VERB-Anteil pro Subkorpus
ax = pos_share["VERB"].sort_values(ascending=False).plot(kind="bar")
ax.set_title("Anteil VERB pro Subkorpus (Stichprobe)")
ax.set_xlabel("Subkorpus")
ax.set_ylabel("Anteil")
plt.show()


## 6. Mini-Übungen

1) Wähle 3 Briefe aus *verschiedenen* Subkorpora und vergleiche:
   - Satzlängen (Tokens pro Satz)
   - POS-Verteilung

2) Finde einen Brief mit vielen `PROPN` (Eigennamen). Was könnte das historisch bedeuten?

3) Suche nach Diskursmarkern wie `sed`, `autem`, `enim` und vergleiche Häufigkeiten über Subkorpora.


In [ ]:
def sentence_lengths(doc):
    return [len([t for t in sent if not t.is_space]) for sent in doc.sents]

docs = list(nlp.pipe(sample["text"].tolist(), batch_size=20))
sent_lens = [l for d in docs for l in sentence_lengths(d)]
pd.Series(sent_lens).describe()
